
**Objetivo:**

Entrenar un modelo Random Forest para predecir el nivel de peligro asociado a casos de violencia
 intrafamiliar y de pareja.

Se evaluarán cuatro escenarios:

- Original
- Balanceado_500
- Balanceado_1000
- Balanceado_2000

El seguimiento experimental se realizará mediante MLflow.

In [0]:
# Importar librerías

#Generales
import os
import json
import tempfile
import time

import pandas as pd
import numpy as np

# MLflow
import mlflow
import mlflow.sklearn

from mlflow.models import infer_signature

# Scikit-learn

import sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    fbeta_score
)

In [0]:
#Configuración MLflow

# Experimento MLflow

NOMBRE_EXPERIMENTO = (
    "/Shared/Prediccion_Nivel_Peligro"
)


mlflow.set_experiment(
    NOMBRE_EXPERIMENTO
)

In [0]:
#Cargar datasets

# Ruta Delta

ruta = (
    "ml_proyecto_7405607705157039.default"
)

train_original = spark.table(
    f"{ruta}.train_original"
)

train_balanceado_500 = spark.table(
    f"{ruta}.train_balanceado_500"
)

train_balanceado_1000 = spark.table(
    f"{ruta}.train_balanceado_1000"
)

train_balanceado_2000 = spark.table(
    f"{ruta}.train_balanceado_2000"
)

test = spark.table(
    f"{ruta}.test"
)

In [0]:
#Cargar pesos de clase

#Pesos de clase

df_pesos = spark.table(
    f"{ruta}.pesos_clases_modelos"
)

In [0]:
pesos_escenarios = {}


escenarios_pesos = (
    df_pesos
    .select("escenario")
    .distinct()
    .toPandas()["escenario"]
)


for escenario in escenarios_pesos:

    datos = (
        df_pesos
        .filter(
            df_pesos.escenario == escenario
        )
        .toPandas()
    )


    pesos_escenarios[escenario] = dict(
        zip(
            datos["clase"],
            datos["peso_clase"]
        )
    )

In [0]:
variables_modelo = [

    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "presunto_agresor_cod"

]

target = "nivel_peligro"

In [0]:
datasets = {

    "original":
        train_original,

    "balanceado_500":
        train_balanceado_500,

    "balanceado_1000":
        train_balanceado_1000,

    "balanceado_2000":
        train_balanceado_2000
}

In [0]:
columnas_requeridas = (
    variables_modelo + [target]
)

for nombre, df in datasets.items():

    faltantes = (
        set(columnas_requeridas)
        -
        set(df.columns)
    )


    if faltantes:

        raise ValueError(
            f"{nombre} tiene columnas faltantes: {faltantes}"
        )


faltantes_test = (
    set(columnas_requeridas)
    -
    set(test.columns)
)


if faltantes_test:

    raise ValueError(
        f"test tiene columnas faltantes: {faltantes_test}"
    )


print(
    "Validación de columnas correcta"
)

In [0]:
# Parámetros Random Forest

PARAMETROS_RF = {

    "n_estimators": 200,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "random_state": 42,
    "n_jobs": -1

}

In [0]:
# Construir Pipeline Random Forest

def construir_pipeline_rf(

    variables_modelo,

    pesos_clase

):
   
    # Preprocesamiento
    
    preprocesador = ColumnTransformer(

        transformers=[

            (

                "categoricas",

                OneHotEncoder(
                    handle_unknown="ignore"
                ),

                variables_modelo

            )

        ]

    )

    # Modelo

    modelo = RandomForestClassifier(

        class_weight=pesos_clase,

        **PARAMETROS_RF

    )

    # Pipeline

    pipeline = Pipeline(

        steps=[

            (

                "encoder",

                preprocesador

            ),


            (

                "modelo",

                modelo

            )

        ]

    )


    return pipeline

In [0]:
#Función calcular métricas

def calcular_metricas(
    y_real,
    y_pred
):

    reporte = classification_report(

        y_real,
        y_pred,
        output_dict=True,
        zero_division=0

    )

# Función: calcular métricas

def calcular_metricas(
    y_real,
    y_pred
):

    from sklearn.metrics import (
        classification_report,
        confusion_matrix,
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        fbeta_score
    )

    # Reporte por clase
    reporte = classification_report(
        y_real,
        y_pred,
        output_dict=True,
        zero_division=0
    )

    # Matriz de confusión
    clases = [
    "Peligro bajo",
    "Peligro moderado",
    "Peligro grave",
    "Peligro extremo"
    ]

    matriz = confusion_matrix(
        y_real,
        y_pred,
        labels=clases
    )

    # Especificidad para Peligro extremo
    
    idx = clases.index("Peligro extremo")

    TP = matriz[idx, idx]

    FN = matriz[idx, :].sum() - TP

    FP = matriz[:, idx].sum() - TP

    TN = matriz.sum() - TP - FN - FP

    specificity_extremo = (
        TN / (TN + FP)
        if (TN + FP) > 0
        else 0
    )

    # Diccionario de métricas
   
    metricas = {

        # Métricas globales
        
        "accuracy":

            accuracy_score(
                y_real,
                y_pred
            ),

        "precision_macro":

            precision_score(
                y_real,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "recall_macro":

            recall_score(
                y_real,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "f1_macro":

            f1_score(
                y_real,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "f2_macro":

            fbeta_score(
                y_real,
                y_pred,
                beta=2,
                average="macro",
                zero_division=0
            ),

        # Clase Peligro extremo
        
        "precision_extremo":

            reporte["Peligro extremo"]["precision"],

        "recall_extremo":

            reporte["Peligro extremo"]["recall"],

        "f1_extremo":

            reporte["Peligro extremo"]["f1-score"],

        "f2_extremo":

            fbeta_score(
                y_real,
                y_pred,
                labels=["Peligro extremo"],
                average="macro",
                beta=2,
                zero_division=0
            ),

        "specificity_extremo":

            specificity_extremo,

        "support_extremo":

            reporte["Peligro extremo"]["support"]

    }

    return metricas, reporte, matriz


In [0]:
# Registro MLflow Random Forest

def registrar_mlflow_rf(

    pipeline,

    escenario,

    metricas,

    reporte,

    matriz,

    tiempo_entrenamiento,

    X_train,

    X_test

):

    # Parámetros

    mlflow.log_param(
        "modelo",
        "Random_Forest"
    )

    mlflow.log_param(
        "escenario",
        escenario
    )


    mlflow.log_param(
        "numero_variables",
        len(variables_modelo)
    )


    mlflow.log_param(
        "n_train",
        len(X_train)
    )


    mlflow.log_param(
        "n_test",
        len(X_test)
    )


    mlflow.log_param(
        "sklearn_version",
        sklearn.__version__
    )


    mlflow.log_params(
        PARAMETROS_RF
    )

    # Métricas
    
    for nombre, valor in metricas.items():

        mlflow.log_metric(

            nombre,

            float(valor)

        )


    mlflow.log_metric(

        "tiempo_entrenamiento",

        tiempo_entrenamiento

    )

    # Signature

    input_example = X_train.head(5)


    signature = infer_signature(

        input_example,

        pipeline.predict(
            input_example
        )

    )

    # Modelo MLflow

    mlflow.sklearn.log_model(

        sk_model=pipeline,

        artifact_path="modelo",

        signature=signature,

        input_example=input_example

    )


    # Artefactos

    with tempfile.TemporaryDirectory() as tmp:


        reporte_path = os.path.join(

            tmp,

            "classification_report.json"

        )


        with open(

            reporte_path,

            "w",

            encoding="utf-8"

        ) as archivo:


            json.dump(

                reporte,

                archivo,

                indent=4,

                ensure_ascii=False

            )


        mlflow.log_artifact(

            reporte_path

        )


        matriz_path = os.path.join(

            tmp,

            "matriz_confusion.csv"

        )


        pd.DataFrame(
            matriz
        ).to_csv(

            matriz_path,

            index=False

        )


        mlflow.log_artifact(

            matriz_path

        )

In [0]:
# Entrenamiento Random Forest

def entrenar_random_forest(

    escenario,

    train_df,

    test_df,

    variables_modelo,

    target,

    pesos_clase

):

    """
    Entrena, evalúa y registra un modelo
    Random Forest para un escenario específico.
    """

    # Spark -> Pandas
   
    train_pd = (
        train_df
        .select(
            variables_modelo + [target]
        )
        .toPandas()
    )


    test_pd = (
        test_df
        .select(
            variables_modelo + [target]
        )
        .toPandas()
    )


    # Separar variables
    
    X_train = train_pd[
        variables_modelo
    ]


    y_train = train_pd[
        target
    ]


    X_test = test_pd[
        variables_modelo
    ]


    y_test = test_pd[
        target
    ]


    # Crear Pipeline
    
    pipeline = construir_pipeline_rf(

        variables_modelo,

        pesos_clase

    )

    # MLflow Run

    with mlflow.start_run(

        run_name=f"RF_{escenario}"

    ) as run:

        run_id = run.info.run_id

        # Entrenamiento

        inicio = time.time()


        pipeline.fit(

            X_train,

            y_train

        )


        tiempo_entrenamiento = (
            time.time()
            -
            inicio
        )

        # Predicción

        y_pred = pipeline.predict(

            X_test

        )

        # Métricas

        metricas, reporte, matriz = calcular_metricas(

            y_test,

            y_pred

        )

        metricas[
            "tiempo_entrenamiento"
        ] = tiempo_entrenamiento

        # Registro MLflow

        registrar_mlflow_rf(

            pipeline=pipeline,

            escenario=escenario,

            metricas=metricas,

            reporte=reporte,

            matriz=matriz,

            tiempo_entrenamiento=tiempo_entrenamiento,

            X_train=X_train,

            X_test=X_test

        )

    # Resultado

    resultado = {

        "run_id":

            run_id,


        "modelo":

            "Random_Forest",


        "escenario":

            escenario,


        **metricas

    }

    return resultado

In [0]:
# Entrenamiento escenarios RF

resultados_rf = []

for escenario, train_df in datasets.items():


    print(
        "=" * 60
    )


    print(
        f"Entrenando Random Forest: {escenario}"
    )


    print(
        "=" * 60
    )

    resultado = entrenar_random_forest(

        escenario=escenario,

        train_df=train_df,

        test_df=test,

        variables_modelo=variables_modelo,

        target=target,

        pesos_clase=pesos_escenarios[escenario]

    )

    resultados_rf.append(
        resultado
    )


print(
    "Entrenamiento Random Forest finalizado"
)

In [0]:
# Resultados RF

df_metricas_rf = pd.DataFrame(
    resultados_rf
)

display(
    df_metricas_rf
)

In [0]:
# Guardar tabla Delta

df_metricas_rf_spark = spark.createDataFrame(
    df_metricas_rf
)

(
    df_metricas_rf_spark
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        f"{ruta}.metricas_random_forest"
    )
)